# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaTaseen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## Signal checks (before building the rule)

Checking two signals my rule will rely on, to confirm they're real before encoding anything.

In [6]:
#Signal 1: position volatility vs. decline
signal1 = con.sql(f"""
    WITH march_data AS (
        SELECT * FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    ),
    bounds AS (SELECT MAX(report_date) AS end_d FROM march_data),
    monthly AS (
        SELECT client_hash_id, content_hash_id,
               STDDEV(gsc_avg_position) AS pos_volatility,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_recent,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_earlier
        FROM march_data f, bounds b
        GROUP BY 1, 2
        HAVING imp_earlier >= 20
    )
    SELECT
        CASE WHEN pos_volatility < 5 THEN 'low_volatility'
             WHEN pos_volatility < 15 THEN 'medium_volatility'
             ELSE 'high_volatility' END AS volatility_bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN imp_recent < 0.8 * imp_earlier THEN 1.0 ELSE 0.0 END) AS pct_declining
    FROM monthly
    GROUP BY 1
    ORDER BY 1
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   volatility_bucket      n  pct_declining
0    high_volatility  19949       0.406837
1     low_volatility  53212       0.348230
2  medium_volatility  38410       0.391643


###Signal 1:
 Position volatility vs. decline: CONFIRMED (weak effect). High-volatility pages declined 40.7% of the time vs. 34.8% for low-volatility pages, a real but modest 6-point gap. This confirms volatility is a legitimate signal, even though the effect size is small. This is the signal behind the refresh/staleness flag.

In [7]:
#Signal 2 — CTR vs. position
signal2 = con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position < 5 THEN 'top_5'
             WHEN gsc_avg_position < 10 THEN 'top_10'
             WHEN gsc_avg_position < 20 THEN 'top_20'
             ELSE 'beyond_20' END AS position_bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS avg_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
        AND gsc_impressions >= 10
    GROUP BY 1
    ORDER BY 1
""").df()
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket       n   avg_ctr
0       beyond_20  446551  0.001307
1          top_10  579035  0.003031
2          top_20  359375  0.002614
3           top_5  762568  0.003591


###Signal 2:
Position vs. CTR: CONFIRMED (strong effect). CTR drops consistently and monotonically as position worsens: top_5 pages average 0.36% CTR, dropping to 0.13% for pages beyond position 20 — nearly a 3x difference. This confirms the assumption behind the CTR-fix flag: ranking position is a real, reliable driver of click-through rate.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

##How My Rule Works

My rule checks two things for each page:


####1.How much the page's search ranking changes over time (volatility).
####2.The page's average search position.

Based on these, each page is placed into one of three groups:

####Refresh Soon:
If the page's ranking changes a lot but it still gets a good amount of traffic, it should be updated because improving it could have a big impact.
####CTR/Metadata Fix:
If the page ranks low in search results but still gets some impressions, it may need a better title or description to encourage more people to click on it.
####Monitor:
If the page has a stable ranking and is performing normally, there is no urgent action needed. It should simply be monitored over time.
##Reason Codes
####High Volatility with Traffic:
The page's ranking changes frequently, but it still receives visitors, so it should be reviewed.
####Poor Position:
The page ranks low in search results, so improving its title or description may help increase clicks.
####Stable:
The page is performing normally, so no immediate action is needed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
# Build the feature base for March (reusing the corrected bounds logic from Signal 1)
rule_data = con.sql(f"""
    WITH march_data AS (
        SELECT * FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS total_impressions,
           SUM(gsc_clicks) AS total_clicks,
           AVG(gsc_avg_position) AS avg_position,
           STDDEV(gsc_avg_position) AS pos_volatility
    FROM march_data
    GROUP BY 1, 2
    HAVING total_impressions >= 50
""").df()

print(f'{len(rule_data):,} pages scored')

# Apply the rule
def apply_rule(row):
    if row['pos_volatility'] > 10 and row['total_impressions'] > 500:
        return 'refresh', 'high_volatility_traffic', row['pos_volatility'] * 10 + row['total_impressions'] / 100
    elif row['avg_position'] > 20 and row['total_impressions'] > 50:
        return 'ctr_metadata_fix', 'poor_position', (row['avg_position'] - 20) + row['total_impressions'] / 200
    else:
        return 'monitor', 'stable', 1

results = rule_data.apply(lambda r: apply_rule(r), axis=1, result_type='expand')
results.columns = ['action', 'reason_code', 'score']
rule_data = pd.concat([rule_data, results], axis=1)

# Rank by score, highest priority first
ranked_queue = rule_data.sort_values('score', ascending=False).reset_index(drop=True)

# Write the CSV
import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(ranked_queue['action'].value_counts())
ranked_queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 pages scored
action
monitor             83945
ctr_metadata_fix    26630
refresh              5539
Name: count, dtype: int64


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,pos_volatility,action,reason_code,score
0,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,1.171558,ctr_metadata_fix,poor_position,985.661674
1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,11.195379,13.556438,refresh,high_volatility_traffic,973.904380
2,client_23a62021009f63c4,content_13ef8874a9a1ef5e,921.0,2.0,47.954034,85.303382,refresh,high_volatility_traffic,862.243818
3,client_e547b89c05043229,content_aa376cef98a5fae8,883.0,0.0,28.535477,84.523897,refresh,high_volatility_traffic,854.068972
4,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,60.0,22.558608,8.376206,ctr_metadata_fix,poor_position,722.093608
5,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,197.0,23.335465,1.739056,ctr_metadata_fix,poor_position,704.115465
6,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,163.0,24.355625,4.073137,ctr_metadata_fix,poor_position,662.890625
7,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,30.769353,0.913488,ctr_metadata_fix,poor_position,572.914353
8,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,73.0,23.888656,2.637183,ctr_metadata_fix,poor_position,556.008656
9,client_23a62021009f63c4,content_3f82b120d085d740,889.0,4.0,19.016715,53.921609,refresh,high_volatility_traffic,548.106088


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked_queue.head(20)
top20[['content_hash_id', 'action', 'reason_code', 'score', 'total_impressions', 'avg_position', 'pos_volatility']]


,content_hash_id,action,reason_code,score,total_impressions,avg_position,pos_volatility
0,content_36e53e9c707674fc,ctr_metadata_fix,poor_position,985.661674,194579.0,32.766674,1.171558
1,content_9c057b66c30a3abb,refresh,high_volatility_traffic,973.904380,83834.0,11.195379,13.556438
2,content_13ef8874a9a1ef5e,refresh,high_volatility_traffic,862.243818,921.0,47.954034,85.303382
3,content_aa376cef98a5fae8,refresh,high_volatility_traffic,854.068972,883.0,28.535477,84.523897
4,content_82e35c4845e6c391,ctr_metadata_fix,poor_position,722.093608,143907.0,22.558608,8.376206
5,content_3df3f32f3fd58dea,ctr_metadata_fix,poor_position,704.115465,140156.0,23.335465,1.739056
6,content_df47d1b976106de4,ctr_metadata_fix,poor_position,662.890625,131707.0,24.355625,4.073137
7,content_bdf60c86117079be,ctr_metadata_fix,poor_position,572.914353,112429.0,30.769353,0.913488
8,content_661a7734f691bef5,ctr_metadata_fix,poor_position,556.008656,110424.0,23.888656,2.637183
9,content_3f82b120d085d740,refresh,high_volatility_traffic,548.106088,889.0,19.016715,53.921609


####CTR Fix:
This page has a low search ranking (around 33) but gets a lot of traffic (194,579 impressions). Improving the title or description could help increase clicks. However, if most visitors are already searching for this page by name, a CTR fix may not make much difference.
####Refresh:
This page has unstable rankings and receives 83,834 impressions. Updating the content could help improve its performance. However, if the ranking changes were caused by a temporary Google update, refreshing the page may not help.
####Refresh:
This page's rankings change a lot, but it only gets 921 impressions. It may not be worth prioritizing because very few people visit it.
Refresh: Similar to the previous page, it has unstable rankings but only 883 impressions. It is probably a low priority because of its small amount of traffic.
####CTR Fix:
 This page has a low ranking (around 23) but still receives 143,907 impressions. A better title or description might improve clicks. However, if users already get the information they need directly from the search results, a CTR fix may have little effect.
####CTR Fix:
This page has a low ranking and 140,156 impressions, while its ranking stays stable. Improving the title or description is a reasonable next step. However, if the content itself is weak, updating metadata alone may not help.
####CTR Fix:
This page has a low ranking and 131,707 impressions. Improving the search snippet could increase clicks. However, if this topic normally has a low click rate, the improvement may be limited.
####CTR Fix:
This page ranks around 31 and receives 112,429 impressions. It is a good candidate for improving its title or description. However, if the keyword is very competitive, its ranking may already be as good as expected.
####CTR Fix:
This page ranks around 24 and has 110,424 impressions. A better search snippet may help attract more clicks. However, users may not need to open the page if the search results already answer their question.
####Refresh:
This page has very unstable rankings but only 889 impressions. Since traffic is low, it may not be worth updating before higher-traffic pages.
####Refresh:
This page has moderate ranking changes and 35,925 impressions. Refreshing the content could improve performance. However, if the changes are seasonal, the page may recover on its own.
####CTR Fix:
This page ranks around 37 and gets 97,378 impressions. Improving its title and description may help increase clicks. However, if the keyword is very difficult, there may be little room for improvement.
####CTR Fix:
This page ranks around 24 with 98,572 impressions. It is a good candidate for a metadata update. However, some topics naturally receive fewer clicks than others.
####Refresh:
This page has noticeable ranking changes and 37,306 impressions. Refreshing the content could be useful. However, if the changes are only normal daily fluctuations, an update may not be necessary.
####CTR Fix:
This page ranks around 22 and receives 94,673 impressions. Improving the search snippet may help increase clicks. However, if competitors dominate the search results, the impact could be limited.
####CTR Fix:
This page ranks around 27 with 91,391 impressions. A metadata update is a reasonable recommendation. However, if this click rate is already normal for this type of page, the improvement may be small.
Refresh: This page has moderate ranking changes and 32,118 impressions. Refreshing the content may improve its performance. However, if the ranking changes were caused by a one-time event, updating the page may not be necessary.
####CTR Fix:
This page ranks around 21 and receives 91,388 impressions. Improving the title or description may help attract more clicks. However, if the search results already show a featured snippet, users may not click even after the update.
####CTR Fix:
This page ranks around 24 and gets 89,982 impressions. A metadata update could improve click-through rates. However, if the page is already performing as well as expected for its topic, the improvement may be limited.
####Refresh:
This page has moderate ranking changes and 30,638 impressions. Refreshing the content is a reasonable recommendation. However, if the changes were caused by a temporary event, the page may improve without any updates.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

####Weak Picks

Some pages were marked as "Refresh" because their rankings changed a lot, but they had less than 1,000 impressions. Compared to the other pages in the top 20, which had much higher traffic, these pages may not be the best priorities. With very little traffic, rankings can change a lot just by chance. In the future, the model should also consider traffic before recommending a page for refresh.

####Leakage Check

The model only uses data from March 2026, such as impressions, clicks, average position, and ranking changes. It does not use any future data or information from later months. This makes the results fair because the model only uses information that would have been available at the time the decision was made.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.